# Validación de Audit Tracer con MIMIC-III Clinical Database Demo v1.4

Sección 7.4 del documento final: tres casos de uso sobre datos clínicos reales y anonimizados
(MIMIC-III Demo, PhysioNet, licencia ODbL 1.0, doi:10.13026/C2HM2Q).

Cada afirmación del documento es una **verificación** con resultado `CUMPLE` o `NO CUMPLE` y su evidencia.
Las mismas verificaciones corren como pruebas en `tests/test_validacion_mimic.py`.

**Cómo se ejecuta el sistema (sin atajos):**

* el servidor central (`app.py`) corre en su propio proceso, con base central, cuentas y correo de alertas propios;
* la científica de datos se autentica una vez con **código de dispositivo** y sus scripts de pandas corren en procesos aparte,
  con `import audit_tracer` como única línea añadida;
* el administrador y el auditor usan el dashboard por HTTP.

**Reloj:** las operaciones del analista se ejecutan con `validacion/reloj.py`, que fija la hora del día
(10:00 y 03:10, hora de Colombia) sobre la **fecha real de ejecución**; la línea base del caso 2 usa los cinco días anteriores.

Requisito: `python validacion/descargar_mimic.py` (descarga y verifica los CSV desde PhysioNet).

In [ ]:
import json, os, sys
from datetime import datetime
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "validacion" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from validacion.casos_mimic import ValidacionMIMIC, carpeta_mimic, COLOMBIA

DATOS = carpeta_mimic()
assert DATOS is not None, "Faltan los CSV: python validacion/descargar_mimic.py"
FECHA = datetime.now(COLOMBIA).date().isoformat()
EVIDENCIA = Path(os.environ.get("VALIDACION_EVIDENCIA_DIR") or RAIZ / "validacion" / "evidencia" / FECHA)
pd.set_option("display.max_colwidth", 120)

def tabla(verificaciones):
    return pd.DataFrame([{"#": v.id, "Afirmación": v.afirmacion, "Estado": v.estado, "Detalle": v.detalle}
                         for v in verificaciones]).style.hide(axis="index")

val = ValidacionMIMIC(EVIDENCIA, DATOS)
print("Datos:", DATOS)
print("Evidencia:", EVIDENCIA)

## Preparación
Servidor central, cuentas (ADMIN, AUDITOR, CIENTIFICO_DATOS), destinatario de alertas (oficial de cumplimiento) y login por código de dispositivo de la científica de datos.

In [ ]:
val.preparar()
prep = json.loads((EVIDENCIA / "preparacion.json").read_text(encoding="utf-8"))
print(json.dumps({k: prep[k] for k in ("dataset", "sha256_datos", "login_dispositivo")}, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in val._info.items()}, indent=2, ensure_ascii=False))

## Caso 1 — Flujo de análisis normal
La científica carga `PATIENTS`, `ADMISSIONS` y `LABEVENTS`, consulta columnas, limpia (fillna, merge, drop, dropna, rename) y exporta el reporte de la cohorte y un anexo de creatinina.

In [ ]:
print((RAIZ / "validacion/scripts/caso1_analisis_cohorte.py").read_text(encoding="utf-8"))

In [ ]:
tabla(val.caso1())

In [ ]:
eventos = pd.DataFrame(val._caso1["eventos"])
eventos["subtipo"] = eventos["motivo_fallo"].str.extract(r"\[(\w+)\]")
eventos[["event_id", "timestamp", "tipo_accion", "subtipo", "dataset_nombre", "filas_exportadas", "nivel_alerta", "motivo_alerta"]]

## Caso 2 — Comportamiento anómalo
Cinco días de trabajo habitual (muestras de 120 a 180 filas en horario laboral) y luego, a las 03:10, la exportación completa de `LABEVENTS` (76 074 filas).

In [ ]:
tabla(val.caso2())

In [ ]:
print(json.dumps(json.loads((EVIDENCIA / "caso2/evaluacion_detector_anomalias.json").read_text(encoding="utf-8")),
                 indent=2, ensure_ascii=False)[:4000])

## Caso 3 — Evidencia para auditoría regulatoria
El auditor filtra por fecha, usuario y tipo de operación, descarga los CSV, verifica el hash de un evento y la prueba de Merkle de su lote.

In [ ]:
tabla(val.caso3())

In [ ]:
pd.read_csv(EVIDENCIA / "caso3/reporte_exportaciones_usuario_periodo.csv")

## Resultado

In [ ]:
val.cerrar()
resumen = val.finalizar()
print(f"Fecha de ejecución: {resumen['entorno']['fecha_ejecucion']}  ·  commit {resumen['entorno']['commit'][:10]}"
      f"  ·  {resumen['entorno']['duracion_segundos']} s")
print(resumen["totales"])
pd.DataFrame([{"#": v["id"], "Caso": v["caso"].split(" — ")[0], "Estado": v["estado"], "Afirmación": v["afirmacion"]}
              for v in resumen["verificaciones"]]).style.hide(axis="index")